# Patient Monitor - Autoencoder pentru Detecția Anomaliilor

Acest notebook antrenează un autoencoder pe datele medicale ale pacienților.
Modelul învață pattern-urile normale și detectează anomalii bazat pe eroarea de reconstrucție.

**Structura:**
1. Încărcarea și explorarea datelor
2. Preprocesarea datelor
3. Construirea autoencoder-ului
4. Antrenarea modelului
5. Evaluarea și detecția anomaliilor
6. Exportul modelului

## 1. Încărcarea și explorarea datelor

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')

In [ ]:
# Încarcă datele din CSV
# Opțiunea 1: Upload direct
from google.colab import files
uploaded = files.upload()  # Selectează medical_data_export.csv

df = pd.read_csv('medical_data_export.csv')
print(f'Total înregistrări: {len(df)}')
print(f'\nColoane: {list(df.columns)}')
print(f'\nTipuri pacienți: {df["patient_type"].value_counts().to_dict()}')
df.head()

In [ ]:
# Statistici descriptive
print('=== Statistici per tip pacient ===')
for ptype in df['patient_type'].unique():
    subset = df[df['patient_type'] == ptype]
    print(f'\n--- {ptype} ({len(subset)} citiri) ---')
    anomalies = subset['is_anomaly'].sum()
    print(f'Anomalii existente: {anomalies} ({anomalies/len(subset)*100:.1f}%)')
    print(subset.describe().round(2))

In [ ]:
# Vizualizare distribuții pentru fiecare tip de pacient
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
fig.suptitle('Distribuția valorilor medicale per tip pacient', fontsize=16)

common_features = ['heart_rate', 'spo_2', 'temperature', 'systolic_bp']

for i, ptype in enumerate(df['patient_type'].unique()):
    subset = df[df['patient_type'] == ptype]
    for j, feat in enumerate(common_features):
        ax = axes[i][j]
        subset[feat].dropna().hist(bins=30, ax=ax, color=['#38bdf8', '#4ade80', '#f87171'][i], alpha=0.7)
        ax.set_title(f'{ptype} - {feat}')
        ax.set_xlabel(feat)

plt.tight_layout()
plt.show()

## 2. Preprocesarea datelor

Fiecare tip de pacient are parametri diferiți:
- **CARDIAC**: heart_rate, spo2, temperature, systolic_bp, diastolic_bp, respiratory_rate, hrv, qt_interval, bnp
- **DIABETES**: heart_rate, spo2, temperature, systolic_bp, diastolic_bp, respiratory_rate, blood_glucose
- **RESPIRATORY**: heart_rate, spo2, temperature, systolic_bp, diastolic_bp, respiratory_rate, fev1, etco2

Antrenăm un autoencoder separat pentru fiecare tip.

In [ ]:
# Definim features per tip de pacient
FEATURES = {
    'CARDIAC': ['heart_rate', 'spo_2', 'temperature', 'systolic_bp', 'diastolic_bp',
                'respiratory_rate', 'hrv', 'qt_interval', 'bnp'],
    'DIABETES': ['heart_rate', 'spo_2', 'temperature', 'systolic_bp', 'diastolic_bp',
                 'respiratory_rate', 'blood_glucose'],
    'RESPIRATORY': ['heart_rate', 'spo_2', 'temperature', 'systolic_bp', 'diastolic_bp',
                    'respiratory_rate', 'fev_1', 'etco_2']
}

print('Features per tip:')
for ptype, feats in FEATURES.items():
    print(f'  {ptype}: {len(feats)} features → {feats}')

In [ ]:
def prepare_data(df, patient_type, features):
    """
    Pregătește datele pentru antrenare:
    1. Filtrează pe tipul de pacient
    2. Selectează doar datele NORMALE (is_anomaly = false) pentru antrenare
    3. Elimină rândurile cu valori lipsă
    4. Normalizează cu MinMaxScaler la [0, 1]
    5. Împarte în train/test
    """
    # Filtrează pe tip
    data = df[df['patient_type'] == patient_type].copy()
    print(f'\n=== {patient_type} ===')
    print(f'Total citiri: {len(data)}')

    # Separă normal vs anomalii
    normal_data = data[data['is_anomaly'] == False][features].dropna()
    anomaly_data = data[data['is_anomaly'] == True][features].dropna()
    print(f'Citiri normale: {len(normal_data)}')
    print(f'Citiri anomalii: {len(anomaly_data)}')

    # Normalizare MinMax [0, 1]
    # Formula: x_norm = (x - x_min) / (x_max - x_min)
    scaler = MinMaxScaler()
    normal_scaled = scaler.fit_transform(normal_data)

    # Împarte date normale: 80% train, 20% test
    X_train, X_test = train_test_split(normal_scaled, test_size=0.2, random_state=42)
    print(f'Train: {len(X_train)}, Test: {len(X_test)}')

    # Prepară și anomaliile (pentru evaluare)
    anomaly_scaled = None
    if len(anomaly_data) > 0:
        anomaly_scaled = scaler.transform(anomaly_data)
        print(f'Anomalii (pentru evaluare): {len(anomaly_scaled)}')

    return X_train, X_test, anomaly_scaled, scaler, normal_data, anomaly_data

# Pregătim datele pentru fiecare tip
datasets = {}
for ptype, feats in FEATURES.items():
    datasets[ptype] = prepare_data(df, ptype, feats)

## 3. Construirea Autoencoder-ului

Arhitectura:
```
Input (n features)
  → Dense(n, ReLU)           # Encoder Layer 1
  → Dense(n//2, ReLU)        # Encoder Layer 2
  → Dense(bottleneck, ReLU)  # Bottleneck (compresie maximă)
  → Dense(n//2, ReLU)        # Decoder Layer 1
  → Dense(n, ReLU)           # Decoder Layer 2
  → Dense(n, sigmoid)        # Output (reconstrucție)
```

**Loss function**: MSE (Mean Squared Error)
```
L = (1/n) · Σ(xᵢ - x̂ᵢ)²
```

**Optimizer**: Adam (adaptive learning rate)

In [ ]:
def build_autoencoder(input_dim, encoding_dim=None):
    """
    Construiește un autoencoder simetric.

    Parametri:
    - input_dim: numărul de features (ex: 7 pentru DIABETES)
    - encoding_dim: dimensiunea bottleneck-ului (default: input_dim // 3)

    Arhitectura:
    Input → Encoder(compresie) → Bottleneck → Decoder(decompresie) → Output
    """
    if encoding_dim is None:
        encoding_dim = max(2, input_dim // 3)  # Minim 2 neuroni în bottleneck

    mid_dim = (input_dim + encoding_dim) // 2  # Strat intermediar

    # Construim modelul secvențial
    model = keras.Sequential([
        # === ENCODER ===
        # Strat 1: Input → mid_dim (prima compresie)
        # z₁ = ReLU(W₁ · x + b₁)
        layers.Dense(input_dim, activation='relu', input_shape=(input_dim,), name='encoder_input'),

        # Strat 2: mid_dim → encoding_dim (compresie maximă = bottleneck)
        # z₂ = ReLU(W₂ · z₁ + b₂)
        layers.Dense(mid_dim, activation='relu', name='encoder_mid'),

        # Bottleneck: reprezentarea comprimată
        layers.Dense(encoding_dim, activation='relu', name='bottleneck'),

        # === DECODER ===
        # Strat 3: encoding_dim → mid_dim (prima decompresie)
        # z₃ = ReLU(W₃ · bottleneck + b₃)
        layers.Dense(mid_dim, activation='relu', name='decoder_mid'),

        # Strat 4: mid_dim → input_dim (reconstrucție completă)
        # x̂ = sigmoid(W₄ · z₃ + b₄)
        layers.Dense(input_dim, activation='relu', name='decoder_expand'),

        # Output: sigmoid pentru a returna valori în [0, 1]
        layers.Dense(input_dim, activation='sigmoid', name='output')
    ])

    # Compilăm cu:
    # - Loss: MSE = (1/n) · Σ(xᵢ - x̂ᵢ)²
    # - Optimizer: Adam (learning rate adaptiv)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mse'  # Mean Squared Error
    )

    return model

# Construim un model pentru fiecare tip de pacient
models = {}
for ptype, feats in FEATURES.items():
    input_dim = len(feats)
    model = build_autoencoder(input_dim)
    models[ptype] = model
    print(f'\n=== Model {ptype} (input: {input_dim} features) ===')
    model.summary()

## 4. Antrenarea modelului

Antrenăm fiecare autoencoder pe datele **normale** ale tipului respectiv.

La fiecare **epocă** (trecere completă prin date):
1. Forward pass: datele trec prin rețea, se calculează output-ul
2. Se calculează loss-ul: `MSE = (1/n) · Σ(xᵢ - x̂ᵢ)²`
3. Backpropagation: se calculează gradienții `∂L/∂W`
4. Se actualizează ponderile: `W = W - η · ∂L/∂W`

In [ ]:
# Antrenăm fiecare model
histories = {}

for ptype in FEATURES.keys():
    print(f'\n{"="*50}')
    print(f'Antrenare model: {ptype}')
    print(f'{"="*50}')

    X_train, X_test, anomaly_scaled, scaler, _, _ = datasets[ptype]
    model = models[ptype]

    # Antrenare
    # - epochs=100: 100 treceri complete prin date
    # - batch_size=32: procesăm 32 exemple la un pas
    # - validation_data: evaluăm pe test set la fiecare epocă
    # - Input = Output (autoencoder învață să reconstruiască input-ul)
    history = model.fit(
        X_train, X_train,  # Input = Target (reconstrucție)
        epochs=100,
        batch_size=32,
        validation_data=(X_test, X_test),
        verbose=0  # Fără output la fiecare epocă
    )

    histories[ptype] = history

    # Afișăm loss-ul final
    final_train_loss = history.history['loss'][-1]
    final_val_loss = history.history['val_loss'][-1]
    print(f'  Train Loss (MSE): {final_train_loss:.6f}')
    print(f'  Val Loss (MSE):   {final_val_loss:.6f}')

print('\n✅ Toate modelele au fost antrenate!')

In [ ]:
# Vizualizare curbe de antrenare (Loss vs Epochs)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Curbe de antrenare - Loss (MSE) vs Epochs', fontsize=14)

colors = {'CARDIAC': '#f87171', 'DIABETES': '#facc15', 'RESPIRATORY': '#38bdf8'}

for i, ptype in enumerate(FEATURES.keys()):
    ax = axes[i]
    history = histories[ptype]

    ax.plot(history.history['loss'], label='Train Loss', color=colors[ptype], linewidth=2)
    ax.plot(history.history['val_loss'], label='Validation Loss', color=colors[ptype],
            linewidth=2, linestyle='--', alpha=0.7)

    ax.set_title(f'{ptype}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Evaluarea și detecția anomaliilor

Calculăm **reconstruction error** pentru fiecare citire:
```
error = MSE(x, x̂) = (1/n) · Σ(xᵢ - x̂ᵢ)²
```

Setăm **threshold** (pragul de anomalie):
```
threshold = μ_error + 2 · σ_error
```

Dacă `error > threshold` → **ANOMALIE**

In [ ]:
# Calculăm threshold-ul și evaluăm pentru fiecare tip
thresholds = {}
results = {}

for ptype in FEATURES.keys():
    print(f'\n{"="*50}')
    print(f'Evaluare: {ptype}')
    print(f'{"="*50}')

    X_train, X_test, anomaly_scaled, scaler, _, _ = datasets[ptype]
    model = models[ptype]

    # 1. Reconstruim datele de test (normale)
    X_test_pred = model.predict(X_test, verbose=0)

    # 2. Calculăm eroarea de reconstrucție per citire
    # error_i = (1/n) · Σ(x_j - x̂_j)² pentru fiecare citire i
    test_errors = np.mean(np.square(X_test - X_test_pred), axis=1)

    # 3. Calculăm threshold: μ + 2σ
    mean_error = np.mean(test_errors)
    std_error = np.std(test_errors)
    threshold = mean_error + 2 * std_error
    thresholds[ptype] = threshold

    print(f'  Eroare medie (date normale): {mean_error:.6f}')
    print(f'  Deviație standard:           {std_error:.6f}')
    print(f'  Threshold (μ + 2σ):          {threshold:.6f}')

    # 4. Verificăm pe date normale - câte sunt detectate greșit ca anomalii?
    normal_detected = np.sum(test_errors > threshold)
    print(f'\n  False positives (normale detectate ca anomalii): {normal_detected}/{len(X_test)} ({normal_detected/len(X_test)*100:.1f}%)')

    # 5. Verificăm pe anomalii reale (dacă există)
    if anomaly_scaled is not None and len(anomaly_scaled) > 0:
        anomaly_pred = model.predict(anomaly_scaled, verbose=0)
        anomaly_errors = np.mean(np.square(anomaly_scaled - anomaly_pred), axis=1)
        anomaly_detected = np.sum(anomaly_errors > threshold)
        print(f'  True positives (anomalii detectate corect): {anomaly_detected}/{len(anomaly_scaled)} ({anomaly_detected/len(anomaly_scaled)*100:.1f}%)')

        results[ptype] = {
            'test_errors': test_errors,
            'anomaly_errors': anomaly_errors,
            'threshold': threshold
        }
    else:
        results[ptype] = {
            'test_errors': test_errors,
            'anomaly_errors': np.array([]),
            'threshold': threshold
        }

In [ ]:
# Vizualizare: distribuția erorilor normale vs anomalii
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Distribuția erorilor de reconstrucție', fontsize=14)

for i, ptype in enumerate(FEATURES.keys()):
    ax = axes[i]
    res = results[ptype]

    # Histogramă date normale
    ax.hist(res['test_errors'], bins=50, alpha=0.7, label='Normal', color='#4ade80', density=True)

    # Histogramă anomalii (dacă există)
    if len(res['anomaly_errors']) > 0:
        ax.hist(res['anomaly_errors'], bins=50, alpha=0.7, label='Anomalii', color='#f87171', density=True)

    # Linie threshold
    ax.axvline(x=res['threshold'], color='#facc15', linestyle='--', linewidth=2, label=f'Threshold: {res["threshold"]:.4f}')

    ax.set_title(f'{ptype}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Reconstruction Error (MSE)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Exemplu detaliat: reconstrucție pentru un pacient
ptype = 'CARDIAC'
X_train, X_test, _, scaler, _, _ = datasets[ptype]
model = models[ptype]
features = FEATURES[ptype]

# Luăm o citire normală și o reconstruim
sample = X_test[0:1]
reconstructed = model.predict(sample, verbose=0)

print(f'=== Exemplu reconstrucție {ptype} ===')
print(f'{"Feature":<20} {"Original":>10} {"Reconstruit":>12} {"Eroare":>10}')
print('-' * 55)
for j, feat in enumerate(features):
    orig = sample[0][j]
    recon = reconstructed[0][j]
    err = (orig - recon) ** 2
    print(f'{feat:<20} {orig:>10.4f} {recon:>12.4f} {err:>10.6f}')

total_mse = np.mean(np.square(sample - reconstructed))
print(f'\nMSE total: {total_mse:.6f}')
print(f'Threshold: {thresholds[ptype]:.6f}')
print(f'Anomalie: {"DA" if total_mse > thresholds[ptype] else "NU"}')

## 6. Exportul modelului

Exportăm:
- Modelele antrenate (format `.keras`)
- Scalerele (parametrii min/max pentru normalizare)
- Threshold-urile calculate

Toate sunt necesare pentru integrarea în aplicația Spring Boot.

In [ ]:
import json
import os

# Creăm directorul de export
os.makedirs('export', exist_ok=True)

# Export config (thresholds + scaler params + features)
config = {}
for ptype in FEATURES.keys():
    X_train, X_test, _, scaler, _, _ = datasets[ptype]

    # Salvăm modelul
    models[ptype].save(f'export/autoencoder_{ptype.lower()}.keras')

    # Salvăm configurația
    config[ptype] = {
        'features': FEATURES[ptype],
        'threshold': float(thresholds[ptype]),
        'scaler_min': scaler.data_min_.tolist(),
        'scaler_max': scaler.data_max_.tolist(),
        'input_dim': len(FEATURES[ptype])
    }

    print(f'✅ {ptype}: model salvat, threshold={thresholds[ptype]:.6f}')

# Salvăm config JSON
with open('export/autoencoder_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f'\n✅ Config salvat în export/autoencoder_config.json')
print(f'\nFișiere generate:')
for f in os.listdir('export'):
    size = os.path.getsize(f'export/{f}')
    print(f'  {f} ({size/1024:.1f} KB)')

In [ ]:
# Convertim modelele în format ONNX (pentru integrare Java/Spring Boot)
!pip install tf2onnx -q
import tf2onnx

for ptype in FEATURES.keys():
    model = models[ptype]
    input_dim = len(FEATURES[ptype])

    # Specificăm input signature
    spec = (tf.TensorSpec((None, input_dim), tf.float32, name='input'),)

    # Convertim
    model_proto, _ = tf2onnx.convert.from_keras(model, input_signature=spec,
                                                 output_path=f'export/autoencoder_{ptype.lower()}.onnx')
    print(f'✅ {ptype}: ONNX exportat')

print('\nFișiere finale:')
for f in sorted(os.listdir('export')):
    size = os.path.getsize(f'export/{f}')
    print(f'  {f} ({size/1024:.1f} KB)')

In [ ]:
# Descarcă fișierele exportate
from google.colab import files
import shutil

# Arhivăm totul
shutil.make_archive('autoencoder_models', 'zip', 'export')
files.download('autoencoder_models.zip')
print('\n📥 Descarcă autoencoder_models.zip și dezarhivează în proiect!')

## Rezumat

### Ce am făcut:
1. **Încărcat** datele medicale din Patient Monitor (CSV export din PostgreSQL)
2. **Preprocesare**: normalizare MinMax, separare normal/anomalii, split train/test
3. **Construit** 3 autoencodere (câte unul per tip pacient: CARDIAC, DIABETES, RESPIRATORY)
4. **Antrenat** pe date normale — modelul învață pattern-urile sănătoase
5. **Evaluat** — calculat threshold bazat pe μ + 2σ al erorilor de reconstrucție
6. **Exportat** modele în format ONNX pentru integrare în Spring Boot

### Cum funcționează detecția:
```
Date noi → Normalizare → Autoencoder → Reconstrucție → MSE → Comparare cu Threshold → Normal/Anomalie
```

### Avantaje față de threshold-uri fixe:
- Învață pattern-ul **individual** al fiecărui tip de pacient
- Detectează **combinații** anormale (ex: puls normal + SpO2 normal dar împreună neobișnuite)
- Se **adaptează** la datele reale, nu la reguli hardcodate